# Earthquake Vision — Malatya hasar segmentasyonu: Colab eğitimi

**Önemli:** CodeOcean dosya sunucusu Colab'dan gelen indirmeleri engelliyor (HTTP 403). Bu yüzden veri **önce kendi bilgisayarınızda** indirilir ve Google Drive üzerinden buraya getirilir.

### Bilgisayarınızda (bir kez)
```bash
cd ~/earthquake-vision && source venv/bin/activate && git pull
python -m src.training.audit_dataset                                  # PASSED olmalı
python -m src.training.prepare_malatya --variant malatya
python -m src.training.prepare_malatya --variant kate7_eval_malatya   # ~400 MB
cd data/raw && zip -qr ~/Desktop/kate_cd_data.zip kate_cd && cd ../..
```
Sonra `kate_cd_data.zip` dosyasını Google Drive'ınızın ana klasörüne (**Drive'ım**) yükleyin.

### Bu not defterinde
1. GPU kontrolü
2. Repoyu klonlama (private repo için token)
3. Paketler
4. Veriyi Drive'dan alma
5. **Denetim kapısı:** repodaki denetim raporu `PASSED` olmalı ve Drive'dan gelen metadata, denetlenen dosyalarla birebir aynı olmalı (SHA-256)
6. YOLO formatına dönüştürme (iki varyant)
7. Ana deney B: 7 şehirle eğit, yalnızca Malatya'da ölç (100 epoch); Deney A isteğe bağlı
8. Eşik seçimi (doğrulama kümesi), test değerlendirmesi, modeli uygulamaya aktarma, eğitim eğrileri, paneller, sonuçları indirme

Her adım hata olursa durur; bir hücre kırmızı hata verirse sonrakine geçmeyin.

**Başlamadan önce:** *Runtime → Change runtime type → T4 GPU* (veya L4/A100).

## 1. GPU kontrolü

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "GPU YOK: Runtime -> Change runtime type -> GPU seçin"

## 2. Repoyu klonla

GitHub → *Settings → Developer settings → Personal access tokens → Fine-grained tokens* → yalnızca bu repo için **Contents: Read-only**. Token gizli sorulur ve not defterine kaydedilmez.

In [ ]:
import getpass, os, subprocess
REPO = "selimhancil/earthquake-damage-segmentation"
os.chdir("/content")
if not os.path.isdir("earthquake-damage-segmentation"):
    token = getpass.getpass("GitHub token: ")
    subprocess.run(["git", "clone", "--depth", "1", f"https://{token}@github.com/{REPO}.git"], check=True)
    del token
else:
    subprocess.run(["git", "-C", "earthquake-damage-segmentation", "pull", "--ff-only"], check=False)
os.chdir("/content/earthquake-damage-segmentation")
!git log --oneline -1

def run(cmd):
    """Komutu çalıştırır, çıktısını CANLI gösterir; başarısız olursa hücreyi durdurur."""
    print("$", cmd, flush=True)
    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="", flush=True)
    returncode = proc.wait()
    if returncode != 0:
        raise SystemExit(f"Komut başarısız (çıkış kodu {returncode}): {cmd}")


## 3. Paketler
Colab'da PyTorch hazır gelir; yalnızca Ultralytics kurulur.

In [ ]:
run("pip install -q ultralytics")
import ultralytics, torch, cv2
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__, "| CUDA", torch.cuda.is_available())

## 4. Veriyi Google Drive'dan al

Drive bağlantı izni istenir. `kate_cd_data.zip` dosyası *Drive'ım* klasöründe olmalı (başka yerdeyse `ZIP` yolunu değiştirin).

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
ZIP = "/content/drive/MyDrive/kate_cd_data.zip"
assert os.path.isfile(ZIP), f"{ZIP} bulunamadı. Bilgisayarınızda oluşturup Drive'ım klasörüne yükleyin."
os.makedirs("data/raw", exist_ok=True)
run(f'unzip -q -o "{ZIP}" -d data/raw')
for variant in ("malatya", "kate7_eval_malatya"):
    path = f"data/raw/kate_cd/selection_{variant}.json"
    print(variant, "hazır" if os.path.isfile(path) else "EKSİK (bilgisayarda prepare çalıştırılmamış)")

## 5. Denetim kapısı

Denetim bilgisayarınızda çalıştırıldı ve raporu repoya eklendi (`reports/data_audit.json`). Burada:
- raporun `PASSED` olduğu,
- Drive'dan gelen metadata dosyalarının denetlenen dosyalarla **aynı** olduğu (SHA-256)
kontrol edilir. Veri yeniden indirilmez (`--offline`); eksik dosya varsa durur.

In [ ]:
import json
audit = json.load(open("reports/data_audit.json"))
assert audit["status"] == "PASSED", "Denetim geçmemiş; eğitim yapılmayacak: " + str(audit["blockers"])
print("Denetim:", audit["status"], "| Malatya:", audit["malatya"]["by_split"])
run("python -m src.training.prepare_malatya --variant malatya --offline")
run("python -m src.training.prepare_malatya --variant kate7_eval_malatya --offline")

## 6. YOLO formatına dönüştürme

In [ ]:
run("python -m src.training.convert_to_yolo --variant malatya")
run("python -m src.training.convert_to_yolo --variant kate7_eval_malatya")

## 7. Ana deney — B: 7 şehirle eğit, yalnızca Malatya'da ölç (100 epoch)

İlk denemede (30 epoch) model hâlâ gelişiyordu; bu yüzden `epochs=100, patience=20`. Epoch satırları aşağıda canlı akar. T4'te yaklaşık 30–40 dakika sürebilir; sekmeyi açık tutun.

In [ ]:
run("python -m src.training.train --variant kate7_eval_malatya --name B_kate7_e100 --epochs 100 --patience 20 --device 0")

### (İsteğe bağlı) Deney A — yalnızca Malatya

İlk denemede kullanılamaz çıktı (53 görüntü yetmedi). Makaledeki karşılaştırma için tekrar istenirse aşağıdaki satırın başındaki `#` işaretini kaldırın.

In [ ]:
# run("python -m src.training.train --variant malatya --name A_malatya_e100 --epochs 100 --patience 20 --device 0")

## 8. Eşik seçimi (doğrulama kümesi) + test değerlendirmesi

1. **Eşik doğrulama kümesinde seçilir** (5 Malatya görüntüsü): mikro Dice'ı en yüksek confidence eşiği. Test kümesi bu adımda hiç kullanılmaz.
2. **Test değerlendirmesi** (4 Malatya görüntüsü): mAP (`conf=0.001`) ve piksel IoU/Dice; seçilen eşik otomatik eklenir, paneller o eşikle çizilir.
3. Model `weights/best_model.pt` olarak dışa aktarılır, önerilen eşik `weights/best_model.meta.json` içine yazılır. Uygulama varsayılan eşiği buradan okur.

In [ ]:
run("python -m src.training.evaluate --weights weights/B_kate7_e100_best.pt --variant kate7_eval_malatya "
    "--name B_kate7_e100 --select-threshold --export-app-weights --device 0")

## 9. Eğitim eğrileri (epoch geçmişi)

In [ ]:
import pandas as pd, glob
from IPython.display import Image, display
for csv_path in sorted(glob.glob("runs/segment/*/results.csv")):
    exp = csv_path.split("/")[-2]
    df = pd.read_csv(csv_path); df.columns = [c.strip() for c in df.columns]
    cols = [c for c in ["epoch", "train/seg_loss", "val/seg_loss", "metrics/precision(M)", "metrics/recall(M)",
                        "metrics/mAP50(M)", "metrics/mAP50-95(M)"] if c in df.columns]
    print("=" * 25, exp, f"({len(df)} epoch)", "=" * 25)
    display(df[cols].tail(10))
    png = f"runs/segment/{exp}/results.png"
    if os.path.isfile(png):
        display(Image(png))

## 10. Özet tablo (yalnızca Malatya test kümesi)

In [ ]:
import json
for path in sorted(glob.glob("reports/evaluation_*.json")):
    if path.endswith("_val.json"):
        continue
    r = json.load(open(path))
    s = r.get("ultralytics_seg", {})
    print("=" * 70)
    print(f"{r['experiment']}  |  n = {r['n_images']}  |  val'de seçilen eşik: {r.get('selected_threshold_from_val')}")
    print(f"  mask mAP50 {s.get('mask_map50')}  mAP50-95 {s.get('mask_map50_95')}  (conf=0.001)")
    for t, m in r["pixel_metrics_by_conf_threshold"].items():
        print(f"  conf>={t}: IoU {m['micro_iou']} {m['micro_iou_ci95']}  Dice {m['micro_dice_f1']}  "
              f"alan hatası {m['area']['relative_area_error']}")
if os.path.isfile("weights/best_model.meta.json"):
    print("\nUygulama modeli:", json.load(open("weights/best_model.meta.json")))

## 11. Örnek paneller (öncesi | sonrası | ground truth | tahmin)

In [ ]:
for path in sorted(glob.glob("reports/figures/B_kate7_e100/*.png")):
    print(path); display(Image(path))

## 12. Sonuçları indir

Zip içindeki `weights/best_model.pt` ve `weights/best_model.meta.json` dosyalarını bilgisayarınızdaki projenin `weights/` klasörüne koyun; uygulama önerilen eşikle açılır.

In [ ]:
run("zip -q -r /content/earthquake_vision_results.zip weights/*_best.pt weights/best_model.pt "
    "weights/best_model.meta.json reports/ runs/segment/*/results.csv runs/segment/*/results.png runs/segment/*/args.yaml")
from google.colab import files
files.download("/content/earthquake_vision_results.zip")